# Explorative Data Analysis

**Summary**:  
The data retrieved via JobTech API (Arbetsförmedlingen / Platsbanken) contains both ads that have been archived and that are active. This is because I want to retrieve as many ads as possible for the year 2026.

The structure of the active and historic ads is almost the same except for the `version_id` recorded for the historic ads: this is the id associated to an add after being archived. 

The total number of ads is 2933. There are no duplicates in `id`, but several ads have the same `title` + `employer` + `description`. I will keep them assuming they are advertising for more than one position at the same time.

Circa 69% of the ads is in Swedish, circa 31% is in English, and less than 1% is unknown. 

There are few missing or very short descriptions but most of the missing information is in the region and municipality.

Some descriptions contain emails, phone numbers and names that have to be masked. For this reason the raw files are not commited to the repo.

The version 31 of the taxonomy related to the occupational group SSYK 2511 (Systemanalytiker och IT-arkitekter) contains 204 skills. It records only hard skills and sometimes, after the comma, there is disambiguation of the skill. This can create noise when trying to match the skill to the text in the ad description.

In [ ]:
import json
import pandas as pd
import re
from langdetect import detect

## Active Ads

Summary:
- Nans in region and municipality  
- No duplicates in "id", but several ads have the same "title" + "employer" + "description". I will keep them assuming they are advertising for more than one position at the same time.
- Only 1 very short description

In [2]:
with open("../data/raw_active_ads_2511.json", encoding="utf-8") as f:
    raw_active = pd.DataFrame(json.load(f))

raw_active_df = raw_active.copy()
raw_active_df.head()

,id,title,employer,municipality,region,occupation,published,deadline,description
0,30523189,PLM Teamcenter,ITMOTIVE in Sweden Aktiebolag,NaN,NaN,"Verksamhetskonsult, IT",2026-01-27T01:04:50,2026-12-26T23:59:59,"PLM Teamcenter Developer, Solution Architect.\..."
1,30896666,Teamchef Data & Analytics till Dizparc,Bravura Sverige AB,Linköping,Östergötlands län,"Affärskonsult, IT",2026-04-13T14:45:16,2026-10-10T23:59:59,Dizparc i Linköping bygger sitt nästa affärsbe...
2,30896809,Senior BI-konsult – forma framtidens datadrivn...,A Hub AB,Karlstad,Värmlands län,"Verksamhetskonsult, IT",2026-04-13T14:58:59,2026-10-10T23:59:59,Vill du arbeta i en roll där du tar fullt ansv...
3,30903986,Till vår kund Invid söker vi Affärssystems kon...,FollowU AB,Jönköping,Jönköpings län,"Verksamhetskonsult, IT",2026-04-14T15:38:38,2026-10-11T23:59:59,Business Central-konsulter till INVID\nVill du...
4,30915541,IT-Specialist Windows Server,Techrytera AB,Stockholm,Stockholms län,IT-arkitekt/Lösningsarkitekt,2026-04-16T14:50:43,2026-10-13T23:59:59,Om oss På Techrytera AB börjar rekryteringen m...


In [3]:
# Shape, cols, len
print(raw_active_df.info())

<class 'pandas.DataFrame'>
RangeIndex: 289 entries, 0 to 288
Data columns (total 9 columns):
 #   Column        Non-Null Count  Dtype
---  ------        --------------  -----
 0   id            289 non-null    str  
 1   title         289 non-null    str  
 2   employer      289 non-null    str  
 3   municipality  276 non-null    str  
 4   region        276 non-null    str  
 5   occupation    289 non-null    str  
 6   published     289 non-null    str  
 7   deadline      289 non-null    str  
 8   description   289 non-null    str  
dtypes: str(9)
memory usage: 20.4 KB
None


In [ ]:
# nans
missing = (
    raw_active_df.isna() |
    raw_active_df.astype(str).apply(lambda col: col.str.strip().eq(""))
).sum()

missing_summary = pd.DataFrame({
    "missing_count": missing,
    "missing_pct": (missing / len(raw_active_df) * 100).round(2)
}).sort_values("missing_pct", ascending=False)

print(missing_summary)

              missing_count  missing_pct
region                   13          4.5
municipality             13          4.5
id                        0          0.0
employer                  0          0.0
title                     0          0.0
occupation                0          0.0
published                 0          0.0
deadline                  0          0.0
description               0          0.0


In [ ]:
# duplicates
print("Duplicated rows based on id:", raw_active_df["id"].duplicated().sum())


# duplicates based on title + employer + description
duplicate_ads = raw_active_df.duplicated(
    subset=["title", "employer", "description"]
).sum()

print("\nDuplicates based on title + employer + description:", duplicate_ads) 


duplicate_texts = raw_active_df[raw_active_df["description"].duplicated(keep=False)]

duplicate_texts[["id","title","employer","published","description"]].head(10)

Duplicated rows based on id: 0

Duplicates based on title + employer + description: 6


,id,title,employer,published,description
14,30981932,Business Consultant M3 - Usability/Extensibili...,Columbus Sweden AB,2026-04-30T15:05:34,Vi söker en tekniskt stark konsult med fokus p...
15,30981944,Business Consultant M3 - Usability/Extensibili...,Columbus Sweden AB,2026-04-30T15:06:16,Vi söker en tekniskt stark konsult med fokus p...
16,30981973,Business Consultant M3 - Usability/Extensibili...,Columbus Sweden AB,2026-04-30T15:07:14,Vi söker en tekniskt stark konsult med fokus p...
17,30990931,Data Engineer - Stockholm,Columbus Sweden AB,2026-05-04T15:46:48,Arbetsuppgifter\n\nSom Data Engineer på Columb...
18,30990940,Data Engineer - Göteborg,Columbus Sweden AB,2026-05-04T15:47:29,Arbetsuppgifter\n\nSom Data Engineer på Columb...
19,30990944,Data Engineer - Malmö,Columbus Sweden AB,2026-05-04T15:48:16,Arbetsuppgifter\n\nSom Data Engineer på Columb...
43,31222965,Integrations- och Lösningsarkitekt –,Adavo AB,2026-06-26T16:39:56,Startdatum: September Slutdatum: Option på för...
44,31222967,Integrations- och Lösningsarkitekt –,Adavo AB,2026-06-26T16:40:00,Startdatum: September Slutdatum: Option på för...
105,31468539,Technical Business Analyst,Cognizant Technology Solutions Sweden AB,2026-09-11T15:32:05,What makes Cognizant a unique place to work? T...
106,31468610,Technical Business Analyst,Cognizant Technology Solutions Sweden AB,2026-09-11T15:32:50,What makes Cognizant a unique place to work? T...


In [6]:
# distribution of titles
unique_titles = raw_active_df["title"].nunique()

print(f"Of {len(raw_active_df)} ads, the unique titles are {unique_titles}.\n")
print(raw_active_df["title"].value_counts().head(10))

Of 289 ads, the unique titles are 259.

title
Kravanalytiker                                        10
Systemarkitekt                                         4
IT-arkitekt                                            4
Senior Enterprise Arkitekt - Business Intelligence     4
Lösningsarkitekt                                       3
Technical Business Analyst                             3
Senior Kravanalytiker                                  2
Integrations- och Lösningsarkitekt –                   2
Mjukvaruarkitekter                                     2
Software Architect                                     2
Name: count, dtype: int64


In [7]:
# Occupatioal group according to SSYK
unique_occupations = raw_active_df["occupation"].nunique()

print(f"Total unique occupations: {unique_occupations}\n")
print(raw_active_df["occupation"].value_counts().head(10))

Total unique occupations: 18

occupation
IT-arkitekt/Lösningsarkitekt       64
Kravanalytiker                     40
Affärskonsult, IT                  31
Systemarkitekt                     28
Enterprise Architect               21
Data scientist                     18
Applikationskonsult                16
Systemanalytiker/Systemutredare    15
IT-arkitekt                        14
Verksamhetskonsult, IT             13
Name: count, dtype: int64


In [8]:
# geographical distribution
print("Nmber of municipalities:", raw_active_df["municipality"].nunique())
print(raw_active_df["municipality"].value_counts().head(10))

print("\nNmber of regions:", raw_active_df["region"].nunique())
print(raw_active_df["region"].value_counts().head(10))

Nmber of municipalities: 44
municipality
Stockholm     110
Göteborg       38
Linköping      15
Malmö          15
Lund            9
Solna           8
Sundbyberg      7
Norrköping      6
Jönköping       5
Örebro          5
Name: count, dtype: int64

Nmber of regions: 19
region
Stockholms län          135
Västra Götalands län     45
Skåne län                27
Östergötlands län        24
Västmanlands län          7
Jönköpings län            5
Örebro län                5
Uppsala län               4
Hallands län              4
Dalarnas län              3
Name: count, dtype: int64


In [9]:
# date distribution
raw_active_df["published"] = pd.to_datetime(raw_active_df["published"])

raw_active_df["year_month"] = raw_active_df["published"].dt.to_period("M")
monthly_ads = raw_active_df["year_month"].value_counts().sort_index()

print(monthly_ads)

year_month
2026-01      1
2026-04     16
2026-05     11
2026-06     18
2026-07     12
2026-08     20
2026-09    158
2026-10     53
Freq: M, Name: count, dtype: int64


In [10]:
# description 
raw_active_df["description_length"] = raw_active_df["description"].str.len()
print("Description lenght:")
print(raw_active_df["description_length"].describe())

print(
    "\nVery short descriptions < 200:",
    (raw_active_df["description_length"] < 200).sum()
)

Description lenght:
count     289.000000
mean     3924.795848
std      1231.171271
min       146.000000
25%      3104.000000
50%      3876.000000
75%      4773.000000
max      6469.000000
Name: description_length, dtype: float64

Very short descriptions < 200: 1


## Historic Ads

Summary:  
- Nans in region and municipality
- No duplicates in "id", but several ads have the same "title" + "employer" + "description". I will keep them assuming they are advertising for more than one position at the same time.
- Only few very short or empty descriptions

In [11]:
with open("../data/raw_historic_ads_2511.json", encoding="utf-8") as f:
    raw_historic = pd.DataFrame(json.load(f))

raw_historic_df = raw_historic.copy()
raw_historic_df.head()

,id,version_id,title,employer,municipality,region,occupation,published,deadline,description
0,30321971,5b04bbf8bb32f20aff74fadd748a875282ce9938,IT-konsult,Capgemini Sverige AB,NaN,NaN,"Affärskonsult, IT",2026-01-01T00:00:00,2026-01-31T23:59:59,Om jobbet\nCapgemini strävar efter att vara en...
1,30321996,a8d242f678bd8a2e52335b17205c33637ab0b33a,Application Consultant,Capgemini Sverige AB,NaN,NaN,"Affärskonsult, IT",2026-01-01T00:00:00,2026-01-31T23:59:59,"Together with a strong, multinational Capgemin..."
2,30321974,bf8e515b7f13898a453328425280cc65639aee87,Sales Representative,Capgemini Sverige AB,NaN,NaN,"Affärskonsult, IT",2026-01-01T00:00:00,2026-01-31T23:59:59,Om jobbet \nCapgemini strävar efter att vara e...
3,30412310,e51b7aa10e5fccc1efcd8072e19b68530d0fd7b2,Dataanalytiker/Data Analyst,UTBETALNINGSMYNDIGHETEN,Stockholm,Stockholms län,"Metodanalytiker, IT",2026-01-02T00:00:00,2026-01-25T23:59:59,Vill du bidra till att lösa en av vår tids stö...
4,30430148,0f84c6915f2ab54294dfa4b1a24af282acc41ade,Chief Architect till MaintMaster!,A Hub AB,Linköping,Östergötlands län,IT-arkitekt/Lösningsarkitekt,2026-01-02T08:21:34,2026-06-21T23:59:59,Vill du arbeta i ett internationellt växande p...


In [12]:
# Shape, cols, len
print(raw_historic_df.info())

<class 'pandas.DataFrame'>
RangeIndex: 2644 entries, 0 to 2643
Data columns (total 10 columns):
 #   Column        Non-Null Count  Dtype
---  ------        --------------  -----
 0   id            2644 non-null   str  
 1   version_id    2644 non-null   str  
 2   title         2644 non-null   str  
 3   employer      2644 non-null   str  
 4   municipality  2482 non-null   str  
 5   region        2482 non-null   str  
 6   occupation    2644 non-null   str  
 7   published     2644 non-null   str  
 8   deadline      2644 non-null   str  
 9   description   2644 non-null   str  
dtypes: str(10)
memory usage: 206.7 KB
None


In [13]:
# nans
missing_historic = (
    raw_historic_df.isna() |
    raw_historic_df.astype(str).apply(lambda col: col.str.strip().eq(""))
).sum()

missing_historic_summary = pd.DataFrame({
    "missing_count": missing_historic,
    "missing_pct": (missing_historic / len(raw_historic_df) * 100).round(2)
}).sort_values("missing_pct", ascending=False)

print(missing_historic_summary)

              missing_count  missing_pct
municipality            162         6.13
region                  162         6.13
description               5         0.19
id                        0         0.00
version_id                0         0.00
employer                  0         0.00
title                     0         0.00
occupation                0         0.00
published                 0         0.00
deadline                  0         0.00


In [14]:
# duplicates
print("Duplicated rows based on id:", raw_historic_df["id"].duplicated().sum())

# duplicates based on title + employer + description
duplicate_historic_ads = raw_historic_df.duplicated(
    subset=["title", "employer", "description"]
).sum()

print("\nDuplicates based on title + employer + description:", duplicate_historic_ads) 

duplicate_historic_texts = raw_historic_df[raw_historic_df["description"].duplicated(keep=False)]
duplicate_historic_texts[["id","title","employer","published","description"]].head(10)

Duplicated rows based on id: 0

Duplicates based on title + employer + description: 193


,id,title,employer,published,description
0,30321971,IT-konsult,Capgemini Sverige AB,2026-01-01T00:00:00,Om jobbet\nCapgemini strävar efter att vara en...
2,30321974,Sales Representative,Capgemini Sverige AB,2026-01-01T00:00:00,Om jobbet \nCapgemini strävar efter att vara e...
17,30440314,Senior uppdragsledande arkitekt inom hospitality,AFRY AB,2026-01-07T13:05:44,Företagsbeskrivning\nAFRY erbjuder tjänster in...
44,30452662,Senior IT-Konsult med fokus Azure sökes till N...,Nordlo Improve AB,2026-01-09T15:12:44,Nordlo Jönköping AB\n\n\r\n\r\nSenior IT-Konsu...
50,30456047,UI/UX designer,GreenBay GreenTech Nordic AB,2026-01-12T09:59:25,
58,30457250,Senior teknisk konsult IFS,Align Consulting AB,2026-01-12T11:52:05,Vi har en stor efterfrågan ifrån våra kunder o...
97,30469932,Payroll Integration Specialist,NXT Interim Göteborg AB,2026-01-14T12:29:15,"Vi söker en ny medarbetare, för uppdrag hos vå..."
107,30473220,System architect hos Crosskey,Barona Professionals AB,2026-01-15T08:37:25,System Architect - Business IT\n📍Mariehamn ell...
109,30474198,Arkitekt Data/Analytics till Northern i Stockholm,Tng Group AB,2026-01-15T10:25:42,"Vill du ha en konsultativ roll med kund, data,..."
135,30486519,Infrastructure Architect,Nobina Sverige AB,2026-01-19T09:23:14,Vill du vara med och forma framtidens IT-infra...


In [15]:
# distribution of titles
unique_historic_titles = raw_historic_df["title"].nunique()

print(f"Of {len(raw_historic_df)} ads, the unique titles are {unique_titles}.\n")
print(raw_historic_df["title"].value_counts().head(15))

Of 2644 ads, the unique titles are 259.

title
Lösningsarkitekt                        46
Kravanalytiker                          45
Data Scientist                          30
Senior Data Scientist                   18
Solution Architect                      16
Senior Kravanalytiker                   16
IT-arkitekt                             15
Enterprise Architect                    13
Senior Lösningsarkitekt                 12
Configuration Manager                   12
Senior Solution Architect               11
Informationsarkitekt                    10
IT-konsult                               9
Sales Representative                     9
Kravanalytiker till Quest Consulting     8
Name: count, dtype: int64


In [16]:
# Occupatioal group according to SSYK
unique_historic_occupations = raw_historic_df["occupation"].nunique()

print(f"Total unique occupations: {unique_historic_occupations}\n")
print(raw_historic_df["occupation"].value_counts().head(10))

Total unique occupations: 16

occupation
IT-arkitekt/Lösningsarkitekt       844
Kravanalytiker                     306
Systemanalytiker/Systemutredare    251
Data scientist                     236
Systemarkitekt                     199
Affärskonsult, IT                  189
Enterprise Architect               156
Verksamhetskonsult, IT             152
Infrastrukturarkitekt               85
IT-arkitekt                         78
Name: count, dtype: int64


In [17]:
# geographical distribution
print("Nmber of municipalities:", raw_historic_df["municipality"].nunique())
print(raw_historic_df["municipality"].value_counts().head(10))

print("\nNmber of regions:", raw_historic_df["region"].nunique())
print(raw_historic_df["region"].value_counts().head(10))

Nmber of municipalities: 93
municipality
Stockholm     984
Göteborg      318
Solna         126
Malmö          96
Linköping      74
Sundbyberg     74
Jönköping      61
Norrköping     61
Lund           52
Luleå          52
Name: count, dtype: int64

Nmber of regions: 21
region
Stockholms län          1272
Västra Götalands län     365
Skåne län                183
Östergötlands län        160
Jönköpings län            66
Norrbottens län           55
Örebro län                55
Västmanlands län          54
Västernorrlands län       43
Kronobergs län            37
Name: count, dtype: int64


In [18]:
# date distribution
raw_historic_df["published"] = pd.to_datetime(raw_historic_df["published"])

raw_historic_df["year_month"] = raw_historic_df["published"].dt.to_period("M")
monthly_historic_ads = raw_historic_df["year_month"].value_counts().sort_index()

print(monthly_historic_ads)

year_month
2026-01    307
2026-02    378
2026-03    324
2026-04    393
2026-05    312
2026-06    304
2026-07    186
2026-08    263
2026-09    172
2026-10      5
Freq: M, Name: count, dtype: int64


In [19]:
# description 
raw_historic_df["description_length"] = raw_historic_df["description"].str.len()
print("Description lenght:")
print(raw_historic_df["description_length"].describe())

print(
    "\nVery short descriptions < 200:",
    (raw_historic_df["description_length"] < 200).sum())

Description lenght:
count    2644.000000
mean     3226.871407
std      1392.548666
min         0.000000
25%      2349.000000
50%      3121.000000
75%      4182.000000
max      6506.000000
Name: description_length, dtype: float64

Very short descriptions < 200: 7


## Active and historic combined

Summary:  
- Circa 69% of the ads is in Swedish, circa 31% is in English, and less than 1% is unknown. 
- The descriptions contain emails, phone numbers and names that have to be masked

### Language check on united df

In [20]:
raw_df_combined = pd.concat([raw_active, raw_historic], axis=0, ignore_index=True) 

print(f"Active ads ({len(raw_active_df)}) + Historic ads ({len(raw_historic_df)}) = {len(raw_active_df) + len(raw_historic_df)}")
print(f"Combined df: {len(raw_df_combined)}")
raw_df_combined.head()

Active ads (289) + Historic ads (2644) = 2933
Combined df: 2933


,id,title,employer,municipality,region,occupation,published,deadline,description,version_id
0,30523189,PLM Teamcenter,ITMOTIVE in Sweden Aktiebolag,NaN,NaN,"Verksamhetskonsult, IT",2026-01-27T01:04:50,2026-12-26T23:59:59,"PLM Teamcenter Developer, Solution Architect.\...",NaN
1,30896666,Teamchef Data & Analytics till Dizparc,Bravura Sverige AB,Linköping,Östergötlands län,"Affärskonsult, IT",2026-04-13T14:45:16,2026-10-10T23:59:59,Dizparc i Linköping bygger sitt nästa affärsbe...,NaN
2,30896809,Senior BI-konsult – forma framtidens datadrivn...,A Hub AB,Karlstad,Värmlands län,"Verksamhetskonsult, IT",2026-04-13T14:58:59,2026-10-10T23:59:59,Vill du arbeta i en roll där du tar fullt ansv...,NaN
3,30903986,Till vår kund Invid söker vi Affärssystems kon...,FollowU AB,Jönköping,Jönköpings län,"Verksamhetskonsult, IT",2026-04-14T15:38:38,2026-10-11T23:59:59,Business Central-konsulter till INVID\nVill du...,NaN
4,30915541,IT-Specialist Windows Server,Techrytera AB,Stockholm,Stockholms län,IT-arkitekt/Lösningsarkitekt,2026-04-16T14:50:43,2026-10-13T23:59:59,Om oss På Techrytera AB börjar rekryteringen m...,NaN


In [21]:
def detect_language(text):
    if not text or not text.strip():
        return "unknown"
    try:
        return detect(text)
    except:
        return "unknown"

raw_df_combined["language"] = raw_df_combined["description"].apply(detect_language)
lng_count = raw_df_combined["language"].value_counts()

language_distribution = pd.DataFrame({
    "count": lng_count,
    "pct": (lng_count / len(raw_df_combined) * 100).round(2)
}).sort_values("pct", ascending=False)

print("Language distribution:")
print(language_distribution)

Language distribution:
          count    pct
language              
sv         2008  68.46
en          920  31.37
unknown       5   0.17


In [22]:
# Checking the "unknown" 
unknown_ads = raw_df_combined[raw_df_combined["language"] == "unknown"]

for _, row in unknown_ads.iterrows():
    print("-" * 50)
    print(f"ID: {row["id"]} - Titel: {row["title"]}")
    print("Text:", row["description"])

--------------------------------------------------
ID: 30456047 - Titel: UI/UX designer
Text: 
--------------------------------------------------
ID: 30600803 - Titel: Dynamics 365 Architect / D365 Solution Architect  16004
Text: 
--------------------------------------------------
ID: 30854632 - Titel: Power BI Consultant (Report Development)
Text: 
--------------------------------------------------
ID: 30961171 - Titel: Senior Data Engineer
Text: 
--------------------------------------------------
ID: 30983013 - Titel: Linux Engineer
Text: 


## Personal details check

In [23]:
patterns = {
    "personnummer": r"(?<!\d)(?:19|20)?\d{6}[-+]?\d{4}(?!\d)",
    "email":        r"[\w.%+-]+@[\w.-]+\.[A-Za-z]{2,}",
    "phone":        r"(?<!\d)(?<!\d )(?:\+46|0046|0)[\s-]?(?:\d[\s-]?){7,10}(?!\d)",
}

ignore = r"\d{2}-\d{2}-\d{4}|\d{5}-\d{4}" # ignores dates

In [24]:
for name, pattern in patterns.items():
    hits = []
    for _, row in raw_df_combined.iterrows():
        for m in re.finditer(pattern, row["description"]):
            if re.fullmatch(ignore, m.group().strip()):
                continue
            start, end = m.span()
            hits.append((row["id"], m.group().strip(), row["description"][max(0, start-60):end+60]))

    print("-" * 50)
    print(f"{name.upper()}: {len(hits)} matches in {len({h[0] for h in hits})} ads")

    # for ad_id, match, context in hits[:3]:
    #     print(f"ID {ad_id} | {match}")
    #     print(f"Context:...{context}...\n")
    # masked this part not to print personal information

--------------------------------------------------
PERSONNUMMER: 0 matches in 0 ads
--------------------------------------------------
EMAIL: 101 matches in 74 ads
--------------------------------------------------
PHONE: 77 matches in 39 ads


## Taxonomi

Summary:
- 204 skills as for version 31
- includes only hard skills
- sometimes, after the comma, there is disambiguation of the skill

In [25]:
with open("../data/taxonomi_2511.json", encoding="utf-8") as f:
    taxonomi_df = pd.DataFrame(json.load(f))

taxonomi_df.head()

,skill_id,skill,headlines
0,13ad_dCC_Tn6,"OOS, styr- och utvecklingsmodell",[Styr- och utvecklingsmodeller]
1,16ZN_3Zp_bYV,"Odoo, molnplattform",[Applikationsplattformar]
2,19PK_uxU_Wzy,"Bower, pakethanterare",[Programmerings- och systemutvecklingsverktyg]
3,1CmG_8hR_igc,Bluetooth-teknik,[Kommunikationsprotokoll]
4,1QYQ_Xka_TQ5,"Microsoft Autopilot, systemutvecklingsverktyg",[Programmerings- och systemutvecklingsverktyg]


In [26]:
# Shape, cols, len
print(taxonomi_df.info())

<class 'pandas.DataFrame'>
RangeIndex: 204 entries, 0 to 203
Data columns (total 3 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   skill_id   204 non-null    str   
 1   skill      204 non-null    str   
 2   headlines  204 non-null    object
dtypes: object(1), str(2)
memory usage: 4.9+ KB
None


In [29]:
taxonomi_skills = taxonomi_df["skill"].tolist()

print(f"Total skills in taxonomi: {len(taxonomi_skills)}.\n")
taxonomi_skills[0:10]

Total skills in taxonomi: 204.



['OOS, styr- och utvecklingsmodell',
 'Odoo, molnplattform',
 'Bower, pakethanterare',
 'Bluetooth-teknik',
 'Microsoft Autopilot, systemutvecklingsverktyg',
 'Office SharePoint Server (MOSS), applikationsplattform',
 'Ledningssystem (ISO 9000-serien)',
 'WAP-teknik',
 'Asset Information Management',
 'HTTP/HTTPS, kommunikationsprotokoll']